In [1]:
# Import necessary libraries
import pandas as pd

# Load lap times dataset
lap_times = pd.read_csv('lap_times.csv')

# Load pit stops data (only required columns: raceId, driverId, lap)
pit_stops = pd.read_csv('pit_stops.csv')[['raceId', 'driverId', 'lap']]

# Load races data (only required columns: raceId, circuitId)
races = pd.read_csv('races.csv')[['raceId', 'circuitId']]

In [2]:
# Merge lap times with race information to include circuit details
lap_times = lap_times.merge(races, on='raceId', how='left')

# Normalize lap times by dividing each lap time by the fastest lap in the same race
lap_times['normalized_time'] = lap_times['milliseconds'] / lap_times.groupby('raceId')['milliseconds'].transform('min')

# Merge with pit stop data to identify which laps included a pit stop
lap_times = lap_times.merge(pit_stops, on=['raceId', 'driverId', 'lap'], how='left', indicator=True)

# Create a boolean column indicating whether a lap was a pit stop
lap_times['is_pit_stop'] = lap_times['_merge'] == 'both'

# Remove the '_merge' column as it's no longer needed
lap_times.drop(columns=['_merge'], inplace=True)

# Create a column to mark if a lap is immediately after a pit stop (pit exit)
lap_times['is_pit_exit'] = lap_times['is_pit_stop'].shift(1, fill_value=False)

# Filter out abnormally slow laps (more than 1.5 times the fastest lap in the race)
lap_times = lap_times[lap_times['normalized_time'] <= 1.5]

In [3]:
lap_times.head()

,raceId,driverId,lap,position,time,milliseconds,circuitId,normalized_time,is_pit_stop,is_pit_exit
0,841,20,1,1,1:38.109,98109,1,1.103005,False,False
1,841,20,2,1,1:33.006,93006,1,1.045634,False,False
2,841,20,3,1,1:32.713,92713,1,1.042340,False,False
3,841,20,4,1,1:32.803,92803,1,1.043352,False,False
4,841,20,5,1,1:32.342,92342,1,1.038169,False,False


In [4]:
# Define features for training
features = ['circuitId', 'lap', 'is_pit_stop', 'is_pit_exit']

# Create lag-based features: previous lap time (lag_1) and the lap before that (lag_2)
lap_times['lag_1'] = lap_times.groupby(['raceId', 'driverId'])['normalized_time'].shift(1)
lap_times['lag_2'] = lap_times.groupby(['raceId', 'driverId'])['normalized_time'].shift(2)

# Add lag features to the feature list
features += ['lag_1', 'lag_2']

# Display updated dataset with new features
lap_times.head()

,raceId,driverId,lap,position,time,milliseconds,circuitId,normalized_time,is_pit_stop,is_pit_exit,lag_1,lag_2
0,841,20,1,1,1:38.109,98109,1,1.103005,False,False,NaN,NaN
1,841,20,2,1,1:33.006,93006,1,1.045634,False,False,1.103005,NaN
2,841,20,3,1,1:32.713,92713,1,1.042340,False,False,1.045634,1.103005
3,841,20,4,1,1:32.803,92803,1,1.043352,False,False,1.042340,1.045634
4,841,20,5,1,1:32.342,92342,1,1.038169,False,False,1.043352,1.042340


In [5]:
# Import required libraries for model training
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Remove rows with missing values in selected features or target variable
lap_times.dropna(subset=features + ['normalized_time'], inplace=True)

# Define input features (X) and target variable (y)
X = lap_times[features]
y = lap_times['normalized_time']

# Split dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('MSE:', mean_squared_error(y_test, y_pred))

MSE: 0.0029837267659994344


In [7]:
from sklearn.ensemble import RandomForestRegressor

In [8]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred)
print('Random Forest MSE:', mse_rf)

y_pred_lr = model.predict(X_test)
mse_lr = mean_squared_error(y_test, y_pred_lr)
print('Linear Regression MSE:', mse_lr)

improvement = mse_lr - mse_rf
print('MSE Improvement:', improvement)

Random Forest MSE: 0.0017692132576299826
Linear Regression MSE: 0.0029837267659994344
MSE Improvement: 0.0012145135083694518


In [9]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)  
y_pred_gb = gb_model.predict(X_test)
mse_gb = mean_squared_error(y_test, y_pred_gb)
print('Gradient Boosting MSE:', mse_gb)

Gradient Boosting MSE: 0.0022758533155337246


In [10]:
import xgboost as xgb

In [11]:
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
print('XGBoost MSE:', mse_xgb)

XGBoost MSE: 0.0018245409018525643


In [12]:
import lightgbm as lgb 

In [13]:
lgb_model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)
mse_lgb = mean_squared_error(y_test, y_pred_lgb)
print('LightGBM MSE:', mse_lgb)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001533 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 644
[LightGBM] [Info] Number of data points in the train set: 429402, number of used features: 6
[LightGBM] [Info] Start training from score 1.062245
LightGBM MSE: 0.001955704860118339


In [14]:
from sklearn.neural_network import MLPRegressor

In [15]:
nn_model = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
nn_model.fit(X_train, y_train)
y_pred_nn = nn_model.predict(X_test)
mse_nn = mean_squared_error(y_test, y_pred_nn)
print('Neural Network MSE:', mse_nn)

Neural Network MSE: 0.0032609137326933056


In [22]:
lap_times.tail(10)

,raceId,driverId,lap,position,time,milliseconds,circuitId,normalized_time,is_pit_stop,is_pit_exit,lag_1,lag_2
575019,1131,858,60,20,1:10.406,70406,70,1.040063,False,False,1.043770,1.045218
575020,1131,858,61,20,1:11.008,71008,70,1.048956,False,False,1.040063,1.043770
575021,1131,858,62,20,1:14.400,74400,70,1.099063,False,False,1.048956,1.040063
575022,1131,858,63,20,1:11.949,71949,70,1.062856,False,False,1.099063,1.048956
575023,1131,858,64,20,1:27.312,87312,70,1.289804,False,False,1.062856,1.099063
575024,1131,858,65,19,1:10.742,70742,70,1.045026,False,False,1.289804,1.062856
575025,1131,858,66,19,1:10.855,70855,70,1.046695,False,False,1.045026,1.289804
575026,1131,858,67,19,1:12.454,72454,70,1.070316,False,False,1.046695,1.045026
575027,1131,858,68,19,1:13.607,73607,70,1.087349,False,False,1.070316,1.046695
575028,1131,858,69,19,1:11.489,71489,70,1.056061,False,False,1.087349,1.070316


In [26]:
def predict_lap_time(race_id, driver_id, circuit_id, lap, is_pit_stop, is_pit_exit):
    # Get previous two lap times for lag features
    prev_laps = lap_times[(lap_times['raceId'] == race_id) & (lap_times['driverId'] == driver_id)]
    
    if lap - 1 in prev_laps['lap'].values:
        lag_1 = prev_laps[prev_laps['lap'] == lap - 1]['normalized_time'].values[0]
    else:
        lag_1 = np.nan  # Default if not available

    if lap - 2 in prev_laps['lap'].values:
        lag_2 = prev_laps[prev_laps['lap'] == lap - 2]['normalized_time'].values[0]
    else:
        lag_2 = np.nan  # Default if not available

    # Create input dataframe
    input_data = pd.DataFrame({
        'circuitId': [circuit_id],
        'lap': [lap],
        'is_pit_stop': [is_pit_stop],
        'is_pit_exit': [is_pit_exit],
        'lag_1': [lag_1],
        'lag_2': [lag_2]
    })

    # Fill missing lag values with mean (or another strategy)
    input_data.fillna(X_train.mean(), inplace=True)

    # Predict using the best model (assuming Random Forest performed best)
    predicted_normalized_time = rf_model.predict(input_data)[0]

    # Convert back to actual time
    min_lap_time = lap_times[lap_times['raceId'] == race_id]['milliseconds'].min()
    predicted_time_ms = predicted_normalized_time * min_lap_time

    # Convert to MM:SS.mmm format
    minutes = int(predicted_time_ms // 60000)  # 1 minute = 60000 ms
    seconds = int((predicted_time_ms % 60000) // 1000)  # Convert remainder to seconds
    milliseconds = int(predicted_time_ms % 1000)  # Remaining milliseconds

    print(f"Predicted Lap Time: {minutes:02}:{seconds:02}.{milliseconds:03}")

# Example usage:
race_id_input = int(input("Enter Race ID: "))
driver_id_input = int(input("Enter Driver ID: "))
circuit_id_input = int(input("Enter Circuit ID: "))
lap_input = int(input("Enter Lap Number: "))
is_pit_stop_input = int(input("Is Pit Stop? (1 for Yes, 0 for No): "))
is_pit_exit_input = int(input("Is Pit Exit? (1 for Yes, 0 for No): "))

predict_lap_time(race_id_input, driver_id_input, circuit_id_input, lap_input, is_pit_stop_input, is_pit_exit_input)


Predicted Lap Time: 01:32.804
